In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense

# Find the dataset folder from the project root or this notebook folder
dataset_root = Path("LP IV Updated Datasets")
if not dataset_root.is_dir():
    dataset_root = Path("../LP IV Updated Datasets")
print("Dataset folder:", dataset_root.resolve())


In [ ]:
# a. Load and preprocess data
data = pd.read_csv(dataset_root / "FNN and CNN/CIFAR10/cifar-10.csv", dtype="uint8")
print("Dataset CSV shape:", data.shape)
print("Dataset preview (3 rows, 8 columns):\n", data.iloc[:3, :8])
print("Dataset label counts:\n", data["label"].value_counts().sort_index())
train, test = train_test_split(data, test_size=0.2)
print("Training split shape:", train.shape)
print("Test split shape:", test.shape)


In [ ]:
# Prepare features and labels
y_train = train.pop("label")
y_test = test.pop("label")

x_train = train.to_numpy("float32").reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1) / 255
x_test = test.to_numpy("float32").reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1) / 255

print("Training features:", x_train.shape, "Training labels:", y_train.shape)
print("Test features:", x_test.shape, "Test labels:", y_test.shape)
print("First 10 training labels:", pd.Series(y_train).head(10).tolist())
print("Scaled training pixel range:", np.asarray(x_train).min(), np.asarray(x_train).max())
print("First image, first 10 scaled pixels:", np.asarray(x_train)[0].reshape(-1)[:10])


In [ ]:
# b. Define CNN architecture
model = Sequential([
    Input(shape=(32, 32, 3)),
    Conv2D(32, 3, activation='relu'),
    MaxPooling2D(),
    Flatten(),
    Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Network architecture:")
model.summary()
print("Optimizer:", type(model.optimizer).__name__)
print("Loss function:", model.loss)


In [ ]:
# c. Train model
history = model.fit(x_train, y_train, epochs=5, batch_size=128)
print("Final training epochs:")
print(pd.DataFrame(history.history).tail(3))


In [ ]:
# d. Evaluate performance
loss, accuracy = model.evaluate(x_test, y_test)
print("Test loss:", loss)
print("Accuracy:", accuracy * 100, "%")

# Inspect five predictions without retraining
probabilities = model.predict(np.asarray(x_test)[:5], verbose=0)
predicted = probabilities.argmax(axis=1)
actual = np.asarray(y_test)[:5]
print("First test image class probabilities:", probabilities[0].round(3))
print("First five predictions:")
print(pd.DataFrame({"True label": actual, "Predicted label": predicted}))
